# TF-IDF Text Feature Extraction (IEMOCAP)


## 1) Environment and Paths


In [1]:
from __future__ import annotations

import os
import re
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import importlib.util


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')


def detect_physical_cores() -> int:
    # Linux-first physical core detection using /proc/cpuinfo.
    cpuinfo = Path('/proc/cpuinfo')
    if cpuinfo.exists():
        pairs: set[tuple[str, str]] = set()
        physical_id = None
        core_id = None
        with cpuinfo.open('r', encoding='utf-8', errors='ignore') as handle:
            for raw in handle:
                line = raw.strip()
                if not line:
                    if physical_id is not None and core_id is not None:
                        pairs.add((physical_id, core_id))
                    physical_id = None
                    core_id = None
                    continue
                if line.startswith('physical id'):
                    physical_id = line.split(':', 1)[1].strip()
                elif line.startswith('core id'):
                    core_id = line.split(':', 1)[1].strip()
        if physical_id is not None and core_id is not None:
            pairs.add((physical_id, core_id))
        if pairs:
            return len(pairs)

    logical = os.cpu_count() or 1
    return max(1, logical // 2)


REPO_ROOT = find_repo_root(Path.cwd())
META_CSV = REPO_ROOT / 'datasets' / 'IEMOCAP' / 'iemocap_full_dataset.csv'
IEMOCAP_ROOT = REPO_ROOT / 'datasets' / 'IEMOCAP'
OUT_DIR = REPO_ROOT / 'extracted_features' / 'text'
OUT_CSV = OUT_DIR / 'tfidf_features.csv'
VOCAB_CSV = OUT_DIR / 'tfidf_vocabulary.csv'

OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_PHYSICAL_CORES = detect_physical_cores()
RESERVED_CORES = 2
CPU_WORKERS = max(1, NUM_PHYSICAL_CORES - RESERVED_CORES)

os.environ['OMP_NUM_THREADS'] = str(CPU_WORKERS)
os.environ['MKL_NUM_THREADS'] = str(CPU_WORKERS)
os.environ['OPENBLAS_NUM_THREADS'] = str(CPU_WORKERS)
os.environ['NUMEXPR_NUM_THREADS'] = str(CPU_WORKERS)

print(f'Repo root: {REPO_ROOT}')
print(f'Metadata CSV: {META_CSV}')
print(f'Output feature CSV: {OUT_CSV}')
print(f'Output vocab CSV: {VOCAB_CSV}')
print(f'Physical cores detected: {NUM_PHYSICAL_CORES}')
print(f'CPU workers for this notebook: {CPU_WORKERS}')

EXCLUDED_EMOTIONS = {'sur', 'fea', 'oth', 'dis'}


Repo root: /home/beka/Speech-Emotion-Recognition
Metadata CSV: /home/beka/Speech-Emotion-Recognition/datasets/IEMOCAP/iemocap_full_dataset.csv
Output feature CSV: /home/beka/Speech-Emotion-Recognition/extracted_features/text/tfidf_features.csv
Output vocab CSV: /home/beka/Speech-Emotion-Recognition/extracted_features/text/tfidf_vocabulary.csv
Physical cores detected: 64
CPU workers for this notebook: 62


## 2) Library Setup and Hardware Detection


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

HAS_TORCH = importlib.util.find_spec('torch') is not None
CUDA_AVAILABLE = False
if HAS_TORCH:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()

HAS_RAPIDS_TFIDF = (
    importlib.util.find_spec('cudf') is not None
    and importlib.util.find_spec('cuml') is not None
)
USE_CUDA_TFIDF = CUDA_AVAILABLE and HAS_RAPIDS_TFIDF

print(f'CUDA available: {CUDA_AVAILABLE}')
print(f'RAPIDS TF-IDF available: {HAS_RAPIDS_TFIDF}')
print(f'Using GPU TF-IDF path: {USE_CUDA_TFIDF}')
if CUDA_AVAILABLE and not HAS_RAPIDS_TFIDF:
    print('CUDA is available, but RAPIDS TF-IDF is not installed. Using CPU TF-IDF path.')


CUDA available: True
RAPIDS TF-IDF available: False
Using GPU TF-IDF path: False
CUDA is available, but RAPIDS TF-IDF is not installed. Using CPU TF-IDF path.


## 3) Transcript Parsing Utilities


In [3]:
LINE_RE = re.compile(r'^(?P<utt>\S+)\s+\[[^\]]+\]:\s*(?P<text>.*)$')


def parse_transcript_file(txt_path: str) -> dict[str, str]:
    file_index: dict[str, str] = {}
    with Path(txt_path).open('r', encoding='utf-8', errors='ignore') as handle:
        for line in handle:
            m = LINE_RE.match(line.strip())
            if not m:
                continue
            utt = m.group('utt')
            text = m.group('text').strip()
            if not text:
                continue
            if utt in file_index:
                file_index[utt] = (file_index[utt] + ' ' + text).strip()
            else:
                file_index[utt] = text
    return file_index


def build_transcript_index(iemocap_dir: Path, max_workers: int) -> dict[str, str]:
    files = sorted(iemocap_dir.glob('Session*/dialog/transcriptions/*.txt'))
    if not files:
        raise FileNotFoundError('No transcript files found under Session*/dialog/transcriptions/*.txt')

    use_workers = max(1, min(max_workers, len(files)))
    if os.name == 'nt' and use_workers > 1:
        print('Windows notebook context detected. Using single-worker transcript parsing for stability.')
        use_workers = 1
    print(f'Parsing {len(files)} transcript files with {use_workers} workers...')
    start = time.perf_counter()

    partial_indexes: list[dict[str, str]] = []
    if use_workers == 1:
        for path in files:
            partial_indexes.append(parse_transcript_file(str(path)))
    else:
        try:
            with ProcessPoolExecutor(max_workers=use_workers) as ex:
                for partial in ex.map(parse_transcript_file, [str(p) for p in files], chunksize=4):
                    partial_indexes.append(partial)
        except Exception as exc:
            print(f'Parallel transcript parsing failed ({type(exc).__name__}: {exc}). Falling back to single-worker parsing.')
            partial_indexes = [parse_transcript_file(str(path)) for path in files]

    index: dict[str, str] = {}
    for partial in partial_indexes:
        for utt, text in partial.items():
            if utt in index:
                index[utt] = (index[utt] + ' ' + text).strip()
            else:
                index[utt] = text

    elapsed = time.perf_counter() - start
    print(f'Transcript indexing complete: {len(index)} utterances ({elapsed:.2f}s)')
    return index


## 4) Load and Filter Metadata


In [4]:
df = pd.read_csv(META_CSV)
print(f'Loaded metadata rows: {len(df):,}')

df['emotion'] = df['emotion'].astype(str).str.strip().str.lower()
df['method'] = df['method'].astype(str).str.strip().str.lower()
df['gender'] = df['gender'].astype(str).str.strip().str.upper()

raw_rows = len(df)
# Keep xxx, exclude selected classes, and enforce agreement for labeled classes.
df = df[~df['emotion'].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df['emotion'] == 'xxx') | (df['agreement'] > 0)].copy()

df['utt_id'] = df['path'].apply(lambda p: Path(p).stem)

print(f'Rows after emotion/agreement filters: {len(df):,} ({len(df)/raw_rows:.2%} retained)')
print('Emotion distribution (top 12):')
print(df['emotion'].value_counts().head(12))
df[['session', 'method', 'gender', 'emotion', 'utt_id']].head()


Loaded metadata rows: 10,039
Rows after emotion/agreement filters: 9,887 (98.49% retained)
Emotion distribution (top 12):
emotion
xxx    2507
fru    1849
neu    1708
ang    1103
sad    1084
exc    1041
hap     595
Name: count, dtype: int64


,session,method,gender,emotion,utt_id
0,1,script,F,neu,Ses01F_script02_1_F000
1,1,script,F,fru,Ses01F_script02_1_F001
2,1,script,F,xxx,Ses01F_script02_1_F002
4,1,script,F,neu,Ses01F_script02_1_F004
5,1,script,F,xxx,Ses01F_script02_1_F005


## 5) Build Transcript Index and Attach Text


In [5]:
transcripts = build_transcript_index(IEMOCAP_ROOT, max_workers=CPU_WORKERS)
df['text'] = df['utt_id'].map(transcripts)

with_text = df['text'].notna() & (df['text'].str.len() > 0)
df_text = df[with_text].copy()

coverage = with_text.mean()
print(f'Rows with non-empty text: {len(df_text):,} / {len(df):,} ({coverage:.2%})')
print(f'Unique utterances with text in index: {len(transcripts):,}')

missing_examples = df.loc[~with_text, 'utt_id'].head(10).tolist()
if missing_examples:
    print('Missing text sample utt_ids:', missing_examples)

df_text[['utt_id', 'emotion', 'text']].head()


Parsing 151 transcript files with 62 workers...


Transcript indexing complete: 10084 utterances (1.74s)
Rows with non-empty text: 9,887 / 9,887 (100.00%)
Unique utterances with text in index: 10,084


,utt_id,emotion,text
0,Ses01F_script02_1_F000,neu,Fine.
1,Ses01F_script02_1_F001,fru,[BREATHING]
2,Ses01F_script02_1_F002,xxx,What?
4,Ses01F_script02_1_F004,neu,That's not your flashlight.
5,Ses01F_script02_1_F005,xxx,It's ours; it's my flashlight too.


## 6) Split Configuration and Corpus Stats


In [6]:
train_sessions = [1, 2, 3, 4]
test_sessions = [5]
train_mask = df_text['session'].isin(train_sessions)

train_rows = int(train_mask.sum())
test_rows = int((~train_mask).sum())
print(f'Train sessions: {train_sessions} -> {train_rows:,} rows')
print(f'Test sessions: {test_sessions} -> {test_rows:,} rows')
print(f"Avg characters per utterance (train): {df_text.loc[train_mask, 'text'].str.len().mean():.2f}")
print(f"Avg characters per utterance (all): {df_text['text'].str.len().mean():.2f}")


Train sessions: [1, 2, 3, 4] -> 7,745 rows
Test sessions: [5] -> 2,142 rows
Avg characters per utterance (train): 58.39
Avg characters per utterance (all): 59.04


## 7) TF-IDF Fit and Transform


In [7]:
tfidf_start = time.perf_counter()

if USE_CUDA_TFIDF:
    from cuml.feature_extraction.text import TfidfVectorizer as CuTfidfVectorizer
    import cudf

    vectorizer = CuTfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
    )

    train_text_gpu = cudf.Series(df_text.loc[train_mask, 'text'].astype(str).tolist())
    all_text_gpu = cudf.Series(df_text['text'].astype(str).tolist())

    print('Fitting RAPIDS TF-IDF on train text...')
    vectorizer.fit(train_text_gpu)
    print('Transforming all rows...')
    matrix_gpu = vectorizer.transform(all_text_gpu)
    matrix = matrix_gpu.get() if hasattr(matrix_gpu, 'get') else matrix_gpu

    terms = vectorizer.get_feature_names() if hasattr(vectorizer, 'get_feature_names') else vectorizer.get_feature_names_out()
    idf_values = vectorizer.idf_ if hasattr(vectorizer, 'idf_') else [None] * len(terms)
else:
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
        dtype=np.float32,
    )

    print('Fitting sklearn TF-IDF on train text...')
    vectorizer.fit(df_text.loc[train_mask, 'text'])
    print('Transforming all rows...')
    matrix = vectorizer.transform(df_text['text'])
    terms = vectorizer.get_feature_names_out()
    idf_values = vectorizer.idf_

tfidf_elapsed = time.perf_counter() - tfidf_start
non_zero = int(matrix.nnz) if hasattr(matrix, 'nnz') else None
density = (non_zero / (matrix.shape[0] * matrix.shape[1])) if non_zero is not None and matrix.shape[0] and matrix.shape[1] else 0.0

print(f'TF-IDF complete in {tfidf_elapsed:.2f}s')
print(f'Matrix shape: {matrix.shape}')
print(f'Vocabulary size: {len(terms):,}')
if non_zero is not None:
    print(f'Non-zero entries: {non_zero:,} (density={density:.6f})')


Fitting sklearn TF-IDF on train text...
Transforming all rows...
TF-IDF complete in 1.13s
Matrix shape: (9887, 10480)
Vocabulary size: 10,480
Non-zero entries: 167,782 (density=0.001619)


## 8) Assemble Output Tables


In [8]:
feature_cols = [f'tfidf_{idx:05d}' for idx in range(matrix.shape[1])]
tfidf_df = pd.DataFrame.sparse.from_spmatrix(matrix, columns=feature_cols)

out = df_text[['path', 'session', 'method', 'gender', 'emotion', 'n_annotators', 'agreement', 'utt_id', 'text']].reset_index(drop=True)
out['split'] = out['session'].apply(lambda value: 'train' if value in set(train_sessions) else 'test')
out = pd.concat([out, tfidf_df], axis=1)

vocab = pd.DataFrame({
    'feature_col': feature_cols,
    'term': terms,
    'idf': idf_values,
})

print(f'Output table shape: {out.shape}')
print(f'Vocabulary table shape: {vocab.shape}')
print('Split counts:')
print(out['split'].value_counts())
vocab.head()


Output table shape: (9887, 10490)
Vocabulary table shape: (10480, 3)
Split counts:
split
train    7745
test     2142
Name: count, dtype: int64


,feature_col,term,idf
0,tfidf_00000,able,7.121718
1,tfidf_00001,able to,7.182343
2,tfidf_00002,about,4.302443
3,tfidf_00003,about all,7.652347
4,tfidf_00004,about being,8.163172


## 9) Save CSV Artifacts


In [9]:
write_start = time.perf_counter()
out.to_csv(OUT_CSV, index=False)
vocab.to_csv(VOCAB_CSV, index=False)
write_elapsed = time.perf_counter() - write_start

print(f'Saved TF-IDF feature CSV: {OUT_CSV}')
print(f'Saved TF-IDF vocabulary CSV: {VOCAB_CSV}')
print(f'Rows: {len(out):,} | TF-IDF columns: {len(vocab):,}')
print(f'CSV write time: {write_elapsed:.2f}s')


Saved TF-IDF feature CSV: /home/beka/Speech-Emotion-Recognition/extracted_features/text/tfidf_features.csv
Saved TF-IDF vocabulary CSV: /home/beka/Speech-Emotion-Recognition/extracted_features/text/tfidf_vocabulary.csv
Rows: 9,887 | TF-IDF columns: 10,480
CSV write time: 1372.48s
